In [24]:
# 1. 挂载 Google Drive (运行后按提示点击授权)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import os
import re
import ast
import torch
import requests
import numpy as np
import pandas as pd
import torch.nn as nn
from io import StringIO
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm

# ==========================================
# 核心超参数配置 (升级至 35M 版本)
# ==========================================
TOP_K_LABELS = 3         
DATA_LIMIT = 30000        

# 💡 核心修改 1：切换为 3500 万参数版本的 ESM-2
MODEL_CHECKPOINT = "facebook/esm2_t12_35M_UR50D"

# 💡 核心修改 2：显存预警调整
# 如果你在运行后续代码时遇到 CUDA Out of Memory，请把这里改成 4
BATCH_SIZE = 4            
EPOCHS = 3                
MAX_SEQ_LENGTH = 1000     

# ==========================================
# 路径配置
# ==========================================
PROJECT_DIR = '/content/drive/MyDrive/Protein_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)
RAW_DATA_PATH = os.path.join(PROJECT_DIR, f"uniprot_raw_{DATA_LIMIT}.csv")
FINAL_DATA_PATH = os.path.join(PROJECT_DIR, f"uniprot_top{TOP_K_LABELS}.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 当前计算设备: {device}")
print(f"🧬 即将加载的模型版本: {MODEL_CHECKPOINT}")

🚀 当前计算设备: cuda
🧬 即将加载的模型版本: facebook/esm2_t12_35M_UR50D


In [26]:
def prepare_top_k_dataset(limit, raw_path, final_path, top_k):
    # 如果最终数据集已存在，直接读取
    if os.path.exists(final_path):
        print(f"📦 检测到已存在的 Top-{top_k} 数据集，直接加载...")
        df = pd.read_csv(final_path)
        df['filtered_go_terms'] = df['filtered_go_terms'].apply(ast.literal_eval)
        return df

    # 1. 自动拉取或读取原始数据
    if os.path.exists(raw_path):
        print(f"📦 加载原始大型数据集: {raw_path}")
        df = pd.read_csv(raw_path)
    else:
        print(f"🌐 正在从 UniProt API 拉取 {limit} 条数据 (可能需要几分钟)...")
        url = "https://rest.uniprot.org/uniprotkb/stream"
        params = {"query": "reviewed:true AND organism_id:9606", "format": "tsv", "fields": "accession,sequence,go", "size": limit}
        response = requests.get(url, params=params)
        df = pd.read_csv(StringIO(response.text), sep='\t')
        df.to_csv(raw_path, index=False)
        print("✅ 原始数据拉取并保存完成。")

    # 2. 基础清洗
    df = df.dropna(subset=['Sequence', 'Gene Ontology (GO)']).copy()
    df = df[df['Sequence'].apply(len) <= MAX_SEQ_LENGTH].copy()
    df['go_terms'] = df['Gene Ontology (GO)'].apply(lambda x: re.findall(r'GO:\d{7}', str(x)))

    # 3. Top-K 核心特征提取
    all_go_terms = [term for terms in df['go_terms'] for term in terms]
    term_counts = Counter(all_go_terms)
    top_k_items = term_counts.most_common(top_k)
    valid_terms = {term for term, count in top_k_items}

    print(f"\n📊 Top {top_k} 保留标签分布:")
    for term, count in top_k_items: print(f"  - {term}: {count} 次")

    # 4. 过滤并保存
    df['filtered_go_terms'] = df['go_terms'].apply(lambda terms: [t for t in terms if t in valid_terms])
    df = df[df['filtered_go_terms'].map(len) > 0].copy()
    df.to_csv(final_path, index=False)
    
    print(f"\n✅ 二次清洗完成！有效样本数: {len(df)}")
    print(f"💾 Top-{top_k} 最终数据集已保存至: {final_path}")
    return df

df_topk = prepare_top_k_dataset(DATA_LIMIT, RAW_DATA_PATH, FINAL_DATA_PATH, TOP_K_LABELS)

📦 加载原始大型数据集: /content/drive/MyDrive/Protein_Project/uniprot_raw_30000.csv

📊 Top 3 保留标签分布:
  - GO:0005634: 4991 次
  - GO:0005829: 4705 次
  - GO:0005886: 4645 次

✅ 二次清洗完成！有效样本数: 11166
💾 Top-3 最终数据集已保存至: /content/drive/MyDrive/Protein_Project/uniprot_top3.csv


In [27]:
class ProteinFunctionDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_length):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            seq, padding='max_length', truncation=True, 
            max_length=self.max_length, return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.float32)
        return item

# 1. 标签多热编码
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(df_topk['filtered_go_terms'])
num_labels = len(mlb.classes_)

# 2. 划分训练/验证集 (80% / 20%)
train_seqs, val_seqs, train_labels, val_labels = train_test_split(
    df_topk['Sequence'].tolist(), y_encoded, test_size=0.2, random_state=42
)

# 3. 封装 DataLoader
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
train_dataset = ProteinFunctionDataset(train_seqs, train_labels, tokenizer, MAX_SEQ_LENGTH)
val_dataset = ProteinFunctionDataset(val_seqs, val_labels, tokenizer, MAX_SEQ_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"🚀 DataLoader 准备就绪！训练集样本: {len(train_seqs)}, 验证集样本: {len(val_seqs)}")

🚀 DataLoader 准备就绪！训练集样本: 8932, 验证集样本: 2234


In [28]:
import torch.nn as nn
from transformers import get_cosine_schedule_with_warmup
from torch.amp import autocast, GradScaler # 引入混合精度

print(f"\n🧠 正在初始化拥有 {num_labels} 个输出节点的 ESM-2 模型...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=num_labels, problem_type="multi_label_classification"
)
model = model.to(device)

# ==========================================
# 🚀 优化一：PyTorch 2.0 计算图编译加速
# ==========================================
# 检查是否支持编译 (部分老旧环境可能不支持)
if hasattr(torch, 'compile'):
    print("⚡ 开启 torch.compile 计算图优化 (注：首个 Epoch 前几步会较慢，请耐心等待编译)...")
    # 针对 ESM-2 模型，默认配置通常最稳妥
    model = torch.compile(model)

# ==========================================
# 🚀 优化二：差分学习率优化器
# ==========================================
optimizer_grouped_parameters = [
    {"params": model.esm.parameters(), "lr": 1e-5},     # 预训练层极小学习率
    {"params": model.classifier.parameters(), "lr": 1e-3} # 新分类头较大学习率
]
optimizer = AdamW(optimizer_grouped_parameters)

# ==========================================
# 🚀 优化三：动态学习率调度器 (Warmup + Cosine)
# ==========================================
# 计算总的训练步数
total_training_steps = len(train_loader) * EPOCHS
# 设置 10% 的步数用于 Warmup 预热
num_warmup_steps = int(total_training_steps * 0.1)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=total_training_steps
)

# ==========================================
# 动态正样本权重 (保持不变)
# ==========================================
total_samples = train_labels.shape[0]
pos_counts = train_labels.sum(axis=0)
neg_counts = total_samples - pos_counts
pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# ==========================================
# 🚀 优化四：初始化混合精度 GradScaler
# ==========================================
scaler = GradScaler('cuda')

print(f"\n🔥 开始全副武装的微调训练 (总步数: {total_training_steps}, 预热步数: {num_warmup_steps})...")
for epoch in range(EPOCHS):
    model.train() 
    total_train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for batch in progress_bar:
        optimizer.zero_grad() # 对于现代 PyTorch，你可以尝试 optimizer.zero_grad(set_to_none=True) 以稍微提升性能
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # ⚡ 开启自动混合精度上下文
        with autocast('cuda', dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
        
        total_train_loss += loss.item()
        
        # ⚡ 混合精度反向传播
        scaler.scale(loss).backward()
        
        # 梯度裁剪需要先 unscale
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # ⚡ 更新参数与学习率
        scaler.step(optimizer)
        scaler.update()
        scheduler.step() # 动态调整学习率
        
        # 获取当前分类头的实时学习率用于显示
        current_lr = scheduler.get_last_lr()[-1]
        progress_bar.set_postfix({
            'loss': f"{loss.item():.4f}", 
            'lr': f"{current_lr:.2e}" # 在进度条中观察学习率的起伏变化
        })
        
    print(f"--- Epoch {epoch+1} 结束 | 平均训练损失: {total_train_loss / len(train_loader):.4f} ---")

# ... 上方的训练循环代码保持不变 ...

# 修改保存路径，增加 35M 标识
final_save_path = os.path.join(PROJECT_DIR, f"esm2_35M_top{TOP_K_LABELS}_optimized_finetuned")
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"\n💾 最终 35M 优化版模型已归档至云盘: {final_save_path}")


🧠 正在初始化拥有 3 个输出节点的 ESM-2 模型...


Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


⚡ 开启 torch.compile 计算图优化 (注：首个 Epoch 前几步会较慢，请耐心等待编译)...

🔥 开始全副武装的微调训练 (总步数: 6699, 预热步数: 669)...


Epoch 1/3 [Train]:   0%|          | 1/2233 [00:54<33:48:41, 54.53s/it, loss=0.7992, lr=1.49e-06]W0417 07:01:38.593000 8733 torch/_dynamo/convert_frame.py:1676] [0/8] torch._dynamo hit config.recompile_limit (8)
W0417 07:01:38.593000 8733 torch/_dynamo/convert_frame.py:1676] [0/8]    function: 'wrapper' (/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:829)
W0417 07:01:38.593000 8733 torch/_dynamo/convert_frame.py:1676] [0/8]    last reason: 0/7: self._modules['esm']._modules['encoder']._modules['layer']._modules['0']._modules['attention']._modules['self']._modules['rotary_embeddings']._seq_len_cached is None  # if seq_len != self._seq_len_cached or self._cos_cached.device != x.device:  # transformers/models/esm/modeling_esm.py:105 in _update_cos_sin_tables
W0417 07:01:38.593000 8733 torch/_dynamo/convert_frame.py:1676] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0417 07:01:38.593000 8733 torch/_dynamo/convert_frame.py:1676] [0/8] To diagno

...

--- Epoch 2 结束 | 平均训练损失: 0.4073 ---


Epoch 3/3 [Train]: 100%|██████████| 2233/2233 [09:08<00:00,  4.07it/s, loss=0.0318, lr=0.00e+00]

--- Epoch 3 结束 | 平均训练损失: 0.3250 ---


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 最终 35M 优化版模型已归档至云盘: /content/drive/MyDrive/Protein_Project/esm2_35M_top3_optimized_finetuned


In [29]:
print("\n🔍 正在验证集上评估模型真实表现...")
model.eval()
all_preds = []
all_labels_list = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="[Eval]"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.sigmoid(logits).cpu().numpy()
        
        all_preds.append(probs)
        all_labels_list.append(labels.cpu().numpy())

all_preds = np.vstack(all_preds)
all_labels_matrix = np.vstack(all_labels_list)

# 动态搜索最佳概率截断阈值 (规避 0.5 陷阱)
best_threshold = 0.5
best_macro_f1 = 0
for th in np.arange(0.1, 0.9, 0.1):
    binary_preds_tmp = (all_preds > th).astype(int)
    f1_tmp = f1_score(all_labels_matrix, binary_preds_tmp, average='macro', zero_division=0)
    if f1_tmp > best_macro_f1:
        best_macro_f1 = f1_tmp
        best_threshold = th

# 使用找到的最佳阈值计算最终指标
binary_preds = (all_preds > best_threshold).astype(int)
macro_f1 = f1_score(all_labels_matrix, binary_preds, average='macro', zero_division=0)
micro_f1 = f1_score(all_labels_matrix, binary_preds, average='micro', zero_division=0)
macro_auprc = average_precision_score(all_labels_matrix, all_preds, average='macro')

print("\n" + "=".center(50, "="))
print(f"🏆 验证集终极评估报告 (Top-{TOP_K_LABELS} 标签)".center(45))
print("=".center(50, "="))
print(f"最佳分类概率阈值: {best_threshold:.2f}")
print(f"Macro F1-Score:   {macro_f1:.4f}")
print(f"Micro F1-Score:   {micro_f1:.4f}")
print(f"Macro AUPRC:      {macro_auprc:.4f}")
print("=".center(50, "="))


🔍 正在验证集上评估模型真实表现...


[Eval]: 100%|██████████| 559/559 [01:56<00:00,  4.81it/s]


            🏆 验证集终极评估报告 (Top-3 标签)           
最佳分类概率阈值: 0.40
Macro F1-Score:   0.8036
Micro F1-Score:   0.8028
Macro AUPRC:      0.8844


In [30]:
print(list(mlb.classes_))

['GO:0005634', 'GO:0005829', 'GO:0005886']


In [32]:
import torch
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# 1. 配置加载路径与环境
# ==========================================
# 指向你的最终优化版模型路径
MODEL_PATH = "/content/drive/MyDrive/Protein_Project/esm2_top10_optimized_finetuned"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("正在加载模型与分词器...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model = model.to(device)
model.eval()

# ==========================================
# 2. 定义标签映射字典
# ==========================================
# ⚠️ 极其重要：请运行 `print(list(mlb.classes_))` 获取你真实的 Top-10 标签顺序
# 然后将它们依次填入下方的列表中替换这里的示例！
TOP_10_GO_TAGS = ['GO:0005634', 'GO:0005829', 'GO:0005886']

def predict_all_protein_functions(sequence, threshold=0.5):
    """
    输入氨基酸序列，输出所有 10 个 GO 标签的预测几率，并按置信度排序展示
    """
    # 1. 序列数字化 (Tokenization)
    inputs = tokenizer(
        sequence, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=1000
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # 2. 模型前向传播
    with torch.no_grad():
        logits = model(**inputs).logits
        # 取出 batch 中第一个（也是唯一一个）样本的概率分布
        probs = torch.sigmoid(logits)[0].cpu().numpy()
    
    # 3. 整理全量预测结果
    results = []
    for idx, go_tag in enumerate(TOP_10_GO_TAGS):
        prob = probs[idx]
        has_function = prob > threshold
        
        results.append({
            "GO 标签 (GO Tag)": go_tag,
            "预测几率 (Probability)": f"{prob * 100:.2f}%",
            "判定结果 (Prediction)": "✅ 具备 (Positive)" if has_function else "❌ 不具备 (Negative)",
            "_raw_prob": prob # 用于后续排序的隐藏列
        })
    
    # 4. 转换为 DataFrame 并按几率从高到低排序，呈现出最专业的分析报告
    df_results = pd.DataFrame(results)
    df_results = df_results.sort_values(by="_raw_prob", ascending=False).drop(columns=["_raw_prob"]).reset_index(drop=True)
    
    return df_results

# ==========================================
# 3. 运行全景测试演示
# ==========================================
test_sequence = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"

# 填入你在验证集中得出的最佳阈值 (best_threshold)，例如 0.35 或 0.4
best_threshold = 0.5

print("\n" + "="*60)
print(f"🧬 蛋白质多标签功能全景分析报告")
print(f"输入序列长度: {len(test_sequence)} AA | 判定阈值: {best_threshold}")
print("="*60)

# 生成预测报告表格
full_report_df = predict_all_protein_functions(test_sequence, threshold=best_threshold)

# 在 Jupyter/Colab 中，直接 print 或是让 df 作为单元格最后一行都会渲染出漂亮的数据表格
print(full_report_df.to_markdown(index=False))

正在加载模型与分词器...


Loading weights:   0%|          | 0/111 [00:00<?, ?it/s]


🧬 蛋白质多标签功能全景分析报告
输入序列长度: 65 AA | 判定阈值: 0.5
| GO 标签 (GO Tag)   | 预测几率 (Probability)   | 判定结果 (Prediction)   |
|:-------------------|:-------------------------|:------------------------|
| GO:0005829         | 80.26%                   | ✅ 具备 (Positive)      |
| GO:0005886         | 25.94%                   | ❌ 不具备 (Negative)    |
| GO:0005634         | 3.56%                    | ❌ 不具备 (Negative)    |
